# Database

## Overview

| DB Type              | Features                                                                                                                                                                                                                                                                                                                   | Use Cases                                                                                                      |
| -------------------- | -------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------- |
| **Columnar**         | Schema Evolution; Column-Oriented Storage; Column-Level Compression; Column-Wise Indexing; Analytical Query Performance; Slice-and-Dice: efficiently analyze data by selecting columns (slice) and breaking it down further (dice), allowing complex queries and aggregations on large datasets with speed and flexibility | Data Warehousing; Analytics; Log Processing                                                                    |
| **Document**         | Efficient Query Performance; Document Versioning; Flexible Schema                                                                                                                                                                                                                                                          | Content Management; IoT                                                                                        |
| **Graph**            | Relationship Focus; Deep Insight                                                                                                                                                                                                                                                                                           | Social Networks; Fraud Detection; LLMs                                                                         |
| **Key-Value**        | Data Partitioning; Simple Data Model; High-Write & Query Performance; Developer Friendly                                                                                                                                                                                                                                   | Caching; Session Store                                                                                         |
| **NewSQL**           | Transactions & ACID; Support SQL                                                                                                                                                                                                                                                                                           | High-Transaction; Real-time                                                                                    |
| **NoSQL**            | Horizontal Scalability; High Availability; Distributed Architecture; Flexible Data Model                                                                                                                                                                                                                                   | Big Data; Real-time                                                                                            |
| **Object-Oriented**  | Complex Querying & Navigation; Complex Data Models; Object Persistence; Encapsulation & Data Abstraction; Object Versioning; Inheritance & Polymorphism                                                                                                                                                                    | Object Persistence                                                                                             |
| **Relational / SQL** | Indexing & Optimization; Security Features; Relationship & Referential Integrity; Structured Data; Transactions & ACID; SQL Support                                                                                                                                                                                        | Online Transaction Processing (OLTP); Online Analytical Processing (OLAP)                                      |
| **Spatial**          | Spatial Types & Indexing; Topology & Network Analysis; Geospatial Query Language; Integration with GIS                                                                                                                                                                                                                     | Geographic Information Systems (GIS); Spatial Analysis                                                         |
| **Time-Series**      | Retention Policies; Efficient Storage; Time-Window Aggregation; High Write & Query Performance                                                                                                                                                                                                                             | Sensor Data; Financial Data; Industrial IoT                                                                    |
| **Vector DB**        | High-dimensional vector storage; Similarity search; Approximate nearest neighbor (ANN) search; Scalability; Metadata management: Stores and manages additional information associated with vectors for filtering and contextualization                                                                                     | Generative AI; Natural language processing (NLP); Recommender systems; Image and video search; Fraud detection |

## Selection

![](./assets/database/db-selection-process.svg)

## Choice

![](./assets/database/db-choice.svg)

## Transactions

| Type     | Definition                                                                                                                                                                                                                                                                    | Priority     | Transaction Failure                   | Data Consistency                   | Use Cases                                       |
| -------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------ | ------------------------------------- | ---------------------------------- | ----------------------------------------------- |
| **ACID** | **Atomicity**: Each transaction is completed or aborted; **Consistency**: Guarantees committed transaction state (store valid data); **Isolation**: Transactions are independent; **Durability**: Committed data is never lost                                                | Consistency  | Entire transaction rolls back         | Guaranteed immediately             | Financial Transactions (Banking, Stock Trading) |
| **BASE** | **Basically Available**: System remains available for read and write operations despite failures; **Soft state**: System may be temporarily inconsistent but eventually becomes consistent; **Eventual consistency**: Over time, all data replicas converge to the same state | Availability | May proceed with eventual consistency | Eventual, but not always immediate | Social Media Platforms                          |

## Locking

### Locking Hierarchy

```mermaid
graph LR

db("Database<br/>Data Query: Shared Lock (S)<br/>Data Manipulation: Shared Lock (S)") e1@--> table(Table)
table("Table<br/>Data Query: Intention Shared Lock (IS)<br/>Data Manipulation: Intention Exclusive or Intent Update") e2@--> page(Page)
page("Page<br/>Data Query: Intention Shared Lock (IS)<br/>Data Manipulation: Intention Exclusive or Intent Update") e3@--> row("Row<br/>Data Query: Shared Lock (S)<br/>Data Manipulation: Exclusive or Update Lock")

e1@{ animate: true }
e2@{ animate: true }
e3@{ animate: true }
```

| Aspect                    | Intent Shared (IS)                                                                              | Exclusive (X)                                                         | Intent Exclusive (IX)                                                                                         | Shared with Intent Exclusive (SIX)                                                                                                                          |
| ------------------------- | ----------------------------------------------------------------------------------------------- | --------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Definition**            | Allows multiple readers but prevents updates. Signals intent to acquire an exclusive lock later | Prevents all other users from accessing the data (reading or writing) | Signals intent to acquire an exclusive lock and prevents other users from acquiring any locks (read or write) | Allows multiple readers but prevents updates. Signals intent to acquire an exclusive lock and prevents other users from acquiring any locks (read or write) |
| **Usage**                 | Initial read access before acquiring X lock; Improve concurrency for read-heavy workloads       | Update operations (write, delete)                                     | Prevent other users from acquiring any locks before acquiring X lock; Useful for long-running transactions    | Similar to IS but signals future X lock acquisition; Useful for scenarios where initial read might be followed by update                                    |
| **Impact on Concurrency** | Improves read concurrency                                                                       | Reduces write concurrency                                             | Blocks all access, impacting overall concurrency                                                              | Improves read concurrency initially, reduces write concurrency later                                                                                        |
| **Scalability**           | Scales well with read-heavy workloads                                                           | May impact performance with high write concurrency                    | May impact performance due to blocking all access                                                             | Can provide a balance between read and write concurrency                                                                                                    |

### Locking Types

<table>
<thead>
    <tr>
    <th>Feature</th>
    <th>Pessimistic Locking</th>
    <th>Optimistic Locking</th>
    </tr>
</thead>
<tbody>
    <tr>
    <td><b>Visualization</b></td>
<td>

```mermaid
sequenceDiagram
autonumber

participant User1
participant Database
participant User2

User1->>Database: Begin Transaction
User2->>Database: Begin Transaction
alt User1 reads record
    Database->>User1: Return record
    alt User2 tries to read/update same record
        Database-->>User2: Record locked
        User2->>Database: Rollback Transaction
    else User2 waits
        activate Database
        Database->>Database: Lock record
        User1->>Database: Update record
        Database-->>User1: Record updated
        Database->>User1: Commit Transaction
        User1-->>Database: Commit acknowledged
        Database->>Database: Release Lock record
        deactivate Database
    end
else User1 waits
    alt User2 reads record
        Database->>User2: Return record
        alt User1 tries to read/update same record
            Database-->>User1: Record locked
            User1->>Database: Rollback Transaction
        else User1 waits
            activate Database
            Database->>Database: Lock record
            User2->>Database: Update record
            Database-->>User2: Record updated
            Database->>User2: Commit Transaction
            User2-->>Database: Commit acknowledged
            Database->>Database: Release Lock record
            deactivate Database
        end
    else User2 waits
        activate Database
        Database->>Database: Lock record
        User1->>Database: Update record
        Database-->>User1: Record updated
        Database->>User1: Commit Transaction
        User1-->>Database: Commit acknowledged
        Database->>Database: Release Lock record
        deactivate Database
    end
end
```

</td>
<td>

```mermaid
sequenceDiagram
autonumber

participant User1
participant Database
participant User2

User1->>Database: Request to read record
Database->>User1: Returns record
alt Record is not locked
    User1->>Database: Request to update record
    Database->>User1: Locks record for User1
    activate Database
    User1->>Database: Send updated record
    Database->>User1: Updates record, unlocks it
    deactivate Database
    Database->>User2: Notification of record update
else Record is locked
    Database->>User1: Sends lock notification
end

User2->>Database: Request to read record
Database->>User2: Returns record
alt Record is not locked
    User2->>Database: Request to update record
    Database->>User2: Locks record for User2
    activate Database
    User2->>Database: Send updated record
    Database->>User2: Updates record, unlocks it
    deactivate Database
    Database->>User1: Notification of record update
else Record is locked
    Database->>User2: Sends lock notification
end
```

</td>
    </tr>
    <tr>
        <td><b>Locking Mechanism</b></td>
        <td>Acquires locks on database records before any read/write operation</td>
        <td>No explicit locks; relies on versioning or timestamps</td>
    </tr>
    <tr>
        <td><b>Transaction Isolation</b></td>
        <td>Guarantees serializability (transactions appear to execute one after another)</td>
        <td>Relies on conflict detection during commit</td>
    </tr>
    <tr>
        <td><b>Concurrency</b></td>
        <td>Lower concurrency due to exclusive access</td>
        <td>Higher concurrency as multiple transactions can read data concurrently</td>
    </tr>
    <tr>
        <td><b>Data Integrity</b></td>
        <td>High; ensures only one transaction modifies data at a time</td>
        <td>Lower; potential for "lost updates" if conflicts occur</td>
    </tr>
    <tr>
        <td><b>Implementation</b></td>
        <td>Database-managed; different lock types (shared, exclusive) available</td>
        <td>Application-level; relies on versioning mechanisms (e.g., version numbers, timestamps) in the database</td>
    </tr>
    <tr>
        <td><b>Error Handling</b></td>
        <td>Rollback transactions that encounter locked records</td>
        <td>Retry transactions that encounter conflicts during commit</td>
    </tr>
    <tr>
        <td><b>Use Cases</b></td>
        <td>
            <ul>
            <li>High contention environments (frequent updates)</li>
            <li>Critical data operations (financial transactions)</li>
            <li>Applications requiring strict data consistency</li>
            </ul>
        </td>
    <td>
        <ul>
            <li>Low contention environments (read-heavy systems)</li>
            <li>Non-critical data updates</li>
            <li>Short-lived transactions</li>
        </ul>
    </td>
    </tr>
</tbody>
</table>

## Theorems

### CAP Theorem

```mermaid
    graph TB

    c(C)
    a(A)
    p(P)

    subgraph s [Pick only 2]
        c e1@--- |CA| a e2@--- |AP| p e3@--- |CP| c
    end

    e1@{ animate: true }
    e2@{ animate: true }
    e3@{ animate: true }
```

- **Consistency (C)**: Ensures that all nodes in the system have the same data at the same time
- **Availability (A)**: Ensures that every request gets a response about whether it was successful or failed
- **Partition Tolerance (P)**: Ensures that the system continues to operate despite network partitions or communication failures

<table>
    <thead>
    <tr>
        <th>Aspect</th>
        <th>AP (Availability & Partition Tolerance)</th>
        <th>CA (Consistency & Availability)</th>
        <th>CP (Consistency & Partition Tolerance)</th>
    </tr>
    </thead>
    <tbody>
    <tr>
        <td><b>Visualization</b></td>
<td>

```mermaid
graph TB

user1(User) e1@<--> |read/write| server1(Server)
user2(User) e2@<--> |read/write| server2(Server)

server1 e3@<--> |read/write| db1[(DB)]
server2 e4@<---> |read/write| db2[(DB)]

db1 e5@<--> |replicate| db2

e1@{ animate: true }
e2@{ animate: true }
e3@{ animate: true }
e4@{ animate: true }
e5@{ animate: true }
```

</td>
<td>

```mermaid
graph TB

user1(User) e1@<--> |read/write| server1(Server)
user2(User) e2@<--> |read/write| server2(Server)

server1 e3@<--> |read/write| db[(DB)]
server2 e4@<--> |read/write| db

e1@{ animate: true }
e2@{ animate: true }
e3@{ animate: true }
e4@{ animate: true }
```

</td>
<td>

```mermaid
graph TB

user1(User) e1@<--> |read/write| server(Server)
user2(User) e2@<--> |read/write| server(Server)

server e3@<--> |read/write| primary[(DB)]
primary e4@<--> |read/write| secondary[(DB)]

e1@{ animate: true }
e2@{ animate: true }
e3@{ animate: true }
e4@{ animate: true }
```

</td>
    </tr>
    <tr>
        <td><b>Definition</b></td>
        <td>Some data may not be consistent</td>
        <td>Network issues might stop the system</td>
        <td>Some data might not be available when a failure happens</td>
    </tr>
    <tr>
        <td><b>Use Cases</b></td>
        <td>Social networks, real-time analytics, recommendation systems</td>
        <td>Financial applications, e-commerce</td>
        <td>Multi-datacenter deployments</td>
    </tr>
    <tr>
        <td><b>Examples</b></td>
        <td>Cassandra, DynamoDB, Riak</td>
        <td>Google Spanner, RDBMS with high availability configurations</td>
        <td>MongoDB with replica sets, BigTable</td>
    </tr>
    </tbody>
</table>

### PACELC Theorem

```mermaid
graph TB

partition(Partition)

subgraph cap [CAP]
    direction TB

    availability(Availability)
    cap_consistency(Consistency)
end

subgraph elc [ELC]
    direction TB

    latency(Latency)
    elc_consistency(Consistency)
end

partition e1@--> |yes| cap
partition e2@--> |no| elc

e1@{ animate: true }
e2@{ animate: true }
```

| Theorem    | Scope                                                                      | Consistency Model                                                   | Latency Consideration                                                                                                |
| ---------- | -------------------------------------------------------------------------- | ------------------------------------------------------------------- | -------------------------------------------------------------------------------------------------------------------- |
| **CAP**    | Focuses on impact of network partitions on consistency and availability    | Binary choice between strong consistency and availability           | Doesn't explicitly consider latency                                                                                  |
| **PACELC** | Broader view, acknowledging trade-offs present even under normal operation | Consistency is treated as a spectrum, offering more nuanced options | Recognizes latency as a critical factor alongside consistency and availability (data replication can impact latency) |
